<!-- 학습 보강 셀 -->

# 01. Basic Pipeline 학습 흐름

이 노트북은 RAG의 최소 실행 단위를 한 번에 연결해 보는 예제입니다.
흐름은 `문서 로드 -> 임베딩 -> VectorStoreIndex 생성 -> QueryEngine 질의 -> 근거 확인` 순서입니다.
각 셀을 실행할 때 지금 다루는 객체가 `Document`, `Index`, `QueryEngine`, `Response` 중 무엇인지 구분하면 전체 구조를 이해하기 쉽습니다.

In [1]:
# LlamaIndex 핵심 패키지 설치
# - llama-index-core: Document, Node, Index, QueryEngine 같은 기본 구성 요소를 제공합니다.
# - 이미 설치되어 있다면 실행하지 않아도 됩니다.
# !pip install llama-index-core

In [2]:
# Ollama LLM 연동 패키지 설치
# - Ollama는 로컬에서 LLM을 실행하는 서버입니다.
# - 실행 전 터미널에서 `ollama serve`가 떠 있어야 합니다.
# - 사용할 모델도 미리 받아야 합니다: `ollama pull gemma2:2b`
# !pip install llama-index-llms-ollama

In [3]:
# 임베딩 모델과 파일 리더 설치
# - nomic-embed-text 임베딩 모델도 미리 받아야 합니다: `ollama pull nomic-embed-text`
# - PDF 파일을 읽기 위해 llama-index-readers-file과 pypdf가 필요합니다.

# sentence-transformers 는 HuggingFace 임베딩 모델을 사용할 때 필요합니다. 이 노트북의 Ollama 임베딩에는 필요하지 않습니다.
# !pip install llama-index-embeddings-ollama llama-index-readers-file pypdf sentence-transformers

<!-- 학습 보강 셀 -->

## 실행 전 준비 체크

이후 셀은 로컬 Ollama 서버와 모델이 준비되어 있어야 정상 실행됩니다.
터미널에서 `ollama serve`, `ollama pull gemma2:2b`, `ollama pull nomic-embed-text`를 먼저 확인하세요.
패키지 설치 셀은 한 번만 실행하면 되고, 커널을 새로 열었을 때는 import 셀부터 다시 실행하면 됩니다.

In [4]:
# LlamaIndex에서 사용할 주요 객체를 불러옵니다.
# - Ollama: 질문에 대한 최종 답변을 생성하는 로컬 LLM 연결 객체
# - OllamaEmbedding: 문서를 벡터로 바꾸는 로컬 임베딩 모델 연결 객체
# - VectorStoreIndex: 문서 벡터를 저장하고 검색할 수 있게 만드는 인덱스
# - SimpleDirectoryReader: 디렉토리 안의 파일을 Document 객체로 읽어오는 로더
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

In [5]:
# OLLAMA_MODEL_PREP_CELL
# Ollama 모델은 pip/requirements.txt로 설치되지 않습니다.
# 이 셀은 노트북 실행 전에 필요한 로컬 Ollama 모델이 있는지 확인하고, 없으면 자동으로 pull 합니다.
import subprocess

OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_LLM_MODEL = 'gemma2:2b'
# 한국어 성능 향상을 위해
# 기존 실습에서 사용하던 nomic-embed-text 임베딩 모델을 사용합니다. Ollama에서 미리 받아야 합니다: `ollama pull nomic-embed-text`
OLLAMA_EMBED_MODEL = 'nomic-embed-text'

def _installed_ollama_models() -> set[str]:
    try:
        result = subprocess.run(
            ['ollama', 'list'],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError('Ollama CLI가 설치되어 있지 않습니다. https://ollama.com 에서 설치하세요.') from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Ollama 서버가 실행 중인지 확인하세요. 터미널에서 `ollama serve`를 실행하세요.') from exc

    names = set()
    for line in result.stdout.splitlines()[1:]:
        parts = line.split()
        if parts:
            names.add(parts[0])
    return names

def ensure_ollama_model(model_name: str) -> None:
    installed = _installed_ollama_models()
    candidates = {model_name}
    if ':' not in model_name:
        candidates.add(f'{model_name}:latest')

    if installed.intersection(candidates):
        print(f'이미 설치됨: {model_name}')
        return

    print(f'Ollama 모델 다운로드 중: {model_name}')
    subprocess.run(['ollama', 'pull', model_name], check=True)

for model_name in [OLLAMA_LLM_MODEL, OLLAMA_EMBED_MODEL]:
    ensure_ollama_model(model_name)

이미 설치됨: gemma2:2b
이미 설치됨: nomic-embed-text


In [6]:
# Ollama 모델 설정
# - model: 답변 생성에 사용할 LLM 이름입니다. 로컬 Ollama에 같은 이름의 모델이 있어야 합니다.
# - request_timeout: PDF 기반 질의는 오래 걸릴 수 있으므로 넉넉하게 둡니다.
# - temperature=0: 같은 질문에 최대한 일관된 답변이 나오도록 설정합니다.
llm = Ollama(
    model=OLLAMA_LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    request_timeout=120,
    temperature=0,
)

# 임베딩 모델 설정
# - 문서 검색 품질은 LLM보다 임베딩 모델에 크게 좌우됩니다.
embed_model = OllamaEmbedding(
    model_name=OLLAMA_EMBED_MODEL, # Ollama에서 사용할 임베딩 모델 이름입니다. (nomic-embed-text)
    base_url=OLLAMA_BASE_URL, # Ollama 서버 URL입니다. (http://localhost:11434)
)

In [7]:
# 문서 불러오기
# - 노트북 기준 상위 폴더의 NewData/pdf_sample1 안에 있는 PDF를 읽습니다.
# - SimpleDirectoryReader는 파일을 LlamaIndex Document 객체 목록으로 변환합니다.
documents = SimpleDirectoryReader(input_dir='../NewData/pdf_sample1').load_data()
print('읽어온 문서 수:', len(documents))

읽어온 문서 수: 23


In [8]:
# 첫 번째 문서의 메타데이터와 앞부분을 확인합니다.
# - 문서가 정상적으로 로드되었는지 확인한 뒤 인덱스를 만드는 순서가 안전합니다.
print(documents[0].metadata)
print(documents[0].text[:500])

{'page_label': '1', 'file_name': '240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf', 'file_path': '/Users/cheng80/Documents/WorkSpace/RAG/NewNote/../NewData/pdf_sample1/240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf', 'file_type': 'application/pdf', 'file_size': 359946, 'creation_date': '2025-12-20', 'last_modified_date': '2025-12-20'}
2024
미국의 인공지능(AI) 정책․전략 현황과 변화 방향     - AI 지배력 강화와 초강대국 유지를 위해 미국은 어떻게 변화하고 있는가?-


<!-- 학습 보강 셀 -->

## Document 확인이 중요한 이유

RAG 오류는 모델보다 데이터 로드 단계에서 시작되는 경우가 많습니다.
본문이 비어 있거나 파일명이 잘못 들어오면, 인덱스는 만들어져도 검색 결과가 엉뚱해집니다.
따라서 인덱스를 만들기 전에 `문서 수`, `metadata`, `text 앞부분`을 확인하는 습관이 중요합니다.

In [9]:
# 문서로부터 벡터 스토어 인덱스 생성
# - 각 Document가 여러 Node로 분할되고, Node별 임베딩 벡터가 생성됩니다.
# - 이 단계에서 Ollama 임베딩 서버가 호출됩니다.
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model,
    show_progress=True,
)

/Users/cheng80/Documents/WorkSpace/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 41/41 [00:02<00:00, 17.96it/s]


<!-- 학습 보강 셀 -->

## 인덱스 생성 단계에서 일어나는 일

`VectorStoreIndex.from_documents()`는 단순 저장 함수가 아닙니다.
내부적으로 문서를 검색 가능한 작은 단위로 나누고, 각 조각을 임베딩 벡터로 변환한 뒤, 질문과 비교할 수 있는 검색 구조를 만듭니다.
이 단계가 느리다면 대부분 임베딩 모델 호출 시간이 원인입니다.

In [10]:
index # 인덱스 객체가 잘 생성되었는지 확인합니다. 

In [11]:
# 쿼리 엔진 생성
# - 검색된 문서 조각을 LLM에 전달해 답변을 생성하는 인터페이스입니다.

query_engine = index.as_query_engine(llm=llm)

# reranker 추가
# (선택사항 : 검색된 문서 조각을 LLM이 답변 생성 전에 재평가해서 더 관련성 높은 조각을 선택하도록 합니다.)
# 질문
# -> 벡터 검색으로 후보 5개 검색
# -> bge-reranker-v2-m3가 질문-문서쌍 5개를 다시 읽고 점수화
# -> 상위 3개만 LLM에 전달

# from llama_index.core.postprocessor import SentenceTransformerRerank

# reranker = SentenceTransformerRerank(
#     model="BAAI/bge-reranker-v2-m3",
#     top_n=3,
# )

# query_engine = index.as_query_engine(
#     llm=llm,
#     similarity_top_k=5,
#     node_postprocessors=[reranker],
# )


2026-06-02 11:29:50,397 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [12]:
# 응답 생성
# - 질문과 관련된 Node를 검색한 뒤, 검색 결과를 바탕으로 LLM이 답변합니다.
query = '미국의 인공지능 정책과 주요 변화에 대해 알려줘'
response = query_engine.query(query)
print(response)

2026-06-02 11:29:50,418 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:30:02,170 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


미국은 인공지능(AI)의 안전성 및 책임 있는 개발과 활용을 강조하는 정책을 추진하고 있습니다. 바이든 대통령은 AI 이용 위험 관리를 우선시하며, 민관 모두가 AI 설명 책임을 완수해야 한다는 방침으로 정부 내부뿐만 아니라 사회 전체의 AI 이용에 관한 포괄적 규칙 만들기를 시도하고 있습니다.  

2023년에는 AI 기술 발전과 관련하여 책임 있는 혁신을 위한 대응 방안 제시와 바이든 행정부는 '인공지능의 안심․안전, 신뢰할 수 있는 개발과 활용에 관한 행정명령'을 서명했습니다. 이 명령은 국방생산법(Defense Production Act)에 따라 특정 AI 모델을 개발하는 기업에게 AI 모델의 안전성 테스트 결과를 정부에 공유할 것을 의무화합니다. 

2023년 11월에는 해리스 부통령이 영국 AI 안전성 정상회의에서 안전하고 책임감 있는 AI 이용을 위해 미국 AI 안전연구소 설립 등을 포함한 새로운 이니셔티브를 발표했습니다.  



<!-- 학습 보강 셀 -->

## 답변만 보지 말고 근거를 같이 확인하기

RAG에서는 최종 답변보다 `어떤 문서 조각을 근거로 답했는지`가 더 중요할 때가 많습니다.
다음 셀의 metadata와 source_nodes를 보면 답변이 실제 문서에 기반했는지, 아니면 관련 없는 조각을 보고 생성됐는지 판단할 수 있습니다.

In [13]:
# 응답 메타데이터 확인
# - 어떤 파일/페이지/노드가 답변 근거로 사용되었는지 추적할 때 활용합니다.
response.metadata

{'11c5f873-ea3e-44ef-bf3e-6615b4171af2': {'page_label': '18',
  'file_name': '240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf',
  'file_path': '/Users/cheng80/Documents/WorkSpace/RAG/NewNote/../NewData/pdf_sample1/240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf',
  'file_type': 'application/pdf',
  'file_size': 359946,
  'creation_date': '2025-12-20',
  'last_modified_date': '2025-12-20'},
 '25e27f4d-b5a4-4994-a673-66f84b184bda': {'page_label': '10',
  'file_name': '240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf',
  'file_path': '/Users/cheng80/Documents/WorkSpace/RAG/NewNote/../NewData/pdf_sample1/240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf',
  'file_type': 'application/pdf',
  'file_size': 359946,
  'creation_date': '2025-12-20',
  'last_modified_date': '2025-12-20'}}

In [14]:
# 검색 근거와 점수 확인
# - score는 검색된 source_node와 질문 사이의 관련도입니다.
# - 값의 해석은 사용하는 벡터 스토어와 검색 방식에 따라 달라질 수 있습니다.
for i, node in enumerate(response.source_nodes, start=1):
    print(f'근거 {i} score: {node.score}')
    print(node.node.text[:300])
    print('-' * 80)

근거 1 score: 0.6537461257114919
이 연구기관은 고등교육기관이나 연방 정부 기관, 산업계와 협력해 윤리적이고 신뢰성 있는 AI 시스템 연구개발과 AI 인재 개발 등을 추진한다는 것이 주요 내용• 기업은 (AI) 제품을 도입․공개하기 전에 안전성을 확보해야 할 기본적 책임이 있다고 강조 2023• 인공지능의 안심․안전, 신뢰할 수 있는 개발과 활용에 관한 행정명령(Executive Order on the Safe, Secure, and Trustworthy Development & Use of AI) (10월)• 바이든 대통령 행정명령• AI 위험을 관리하는 데 미국
--------------------------------------------------------------------------------
근거 2 score: 0.6480252811770221
미국의 인공지능(AI) 정책 전략 현황과 변화 방향 
8
 AI 이용에서 안전․안심․신뢰․책임을 강조한 바이든 행정부의 노력 ○ (바이든 행정부) ’23년 한 해 AI 이용의 위험을 지적하고, 민관이 모두 AI 설명 책임을 완수해야 한다는 방침으로 정부 내부뿐만 아니라 사회 전체의 AI 이용에 관한 포괄적 룰 만들기를 시도 - 바이든 대통령은 AI가 가져올 기회를 잡기 위해서는 우선 위험 관리가 중요하다고 명시하고, 개인․사회․보안․경제에 대한 위험을 관리하는 책임 있는 AI 혁신을 촉진하기 위한 정책에 집중 - ’23년 5월에 A
--------------------------------------------------------------------------------
